In [2]:
from langchain_openai import OpenAI
import os

from dotenv import load_dotenv
load_dotenv()

llm = OpenAI(model='gpt-3.5-turbo-instruct', temperature=0.9)

In [3]:
text = "Suggest a personalized workout routine for someone looking to improve cardiovascular endurance and prefers outdoor activities."

print(llm.invoke(text))

 

Monday:
- Warm up with a 10 minute jog or walk
- 30 minutes of interval training on a running trail: alternate between running at a comfortable pace for 3 minutes and sprinting for 1 minute.
- Cool down with a 5 minute walk.

Tuesday:
- 45 minute bike ride on a scenic route
- Incorporate some hill climbs to increase intensity and challenge the leg muscles
- Stretch for 10 minutes after the ride.

Wednesday:
- Warm up with a 10 minute jog or walk
- Do a 20 minute circuit workout at a nearby park: jumping jacks, push-ups, burpees, mountain climbers, and lunges.
- Follow with a 15 minute jog around the park to cool down.

Thursday:
- 1 hour hike on a moderate trail
- Focus on maintaining a steady pace and incorporate some inclines to challenge the heart rate.
- Stretch for 10 minutes after the hike.

Friday:
- Rest day or light yoga/stretching session to allow the body to recover.

Saturday:
- Warm up with a 10 minute jog or walk
- 45 minute outdoor boot camp class: includes body weigh

In [11]:
file = '../../data/AutoPolicy.pdf'

from filesplitters import FileSplitter

splitter = FileSplitter(file)
docs = splitter.pdffilesplitter(splitter_type='recursive')

documents = [doc.page_content for doc in docs]

from langchain.embeddings import OpenAIEmbeddings
embeddings = OpenAIEmbeddings()
# doc_vectors = embeddings.embed_documents(documents)

from langchain_postgres import PGVector

connection="postgresql+psycopg://postgres@localhost:5432/postgres"

pgvector = PGVector(
    embeddings=embeddings,
    collection_name ='my_docs',
    connection = connection,
    use_jsonb=True
)

pgvector


In [8]:
from langchain_core.documents import Document

docs = [
    Document(
        page_content="there are cats in the pond",
        metadata={"id": 1, "location": "pond", "topic": "animals"},
    ),
    Document(
        page_content="ducks are also found in the pond",
        metadata={"id": 2, "location": "pond", "topic": "animals"},
    ),
    Document(
        page_content="fresh apples are available at the market",
        metadata={"id": 3, "location": "market", "topic": "food"},
    ),
    Document(
        page_content="the market also sells fresh oranges",
        metadata={"id": 4, "location": "market", "topic": "food"},
    ),
    Document(
        page_content="the new art exhibit is fascinating",
        metadata={"id": 5, "location": "museum", "topic": "art"},
    ),
    Document(
        page_content="a sculpture exhibit is also at the museum",
        metadata={"id": 6, "location": "museum", "topic": "art"},
    ),
    Document(
        page_content="a new coffee shop opened on Main Street",
        metadata={"id": 7, "location": "Main Street", "topic": "food"},
    ),
    Document(
        page_content="the book club meets at the library",
        metadata={"id": 8, "location": "library", "topic": "reading"},
    ),
    Document(
        page_content="the library hosts a weekly story time for kids",
        metadata={"id": 9, "location": "library", "topic": "reading"},
    ),
    Document(
        page_content="a cooking class for beginners is offered at the community center",
        metadata={"id": 10, "location": "community center", "topic": "classes"},
    ),
]

pgvector.add_documents(docs, ids=[doc.metadata["id"] for doc in docs])

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

## Chains

In LangChain, a chain is an end-to-end wrapper around multiple individual components

Common chain is LLMChain which consists of PromptTemplate -> Model -> OutputParser

The LLMChain works as follows:

* Takes (multiple) input variables.
* Uses the PromptTemplate to format the input variables into a prompt.
* Passes the formatted prompt to the model (LLM or ChatModel).
* If an output parser is provided, it uses the OutputParser to parse the output of the LLM into a final format.

In [13]:
from langchain_core.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain_openai import ChatOpenAI

prompt = PromptTemplate(
            input_variables=['product'],
            template='what is a good name of company that makes the {product}?',)

chain = LLMChain(llm=llm, prompt=prompt)

print(chain.invoke("eco-friendly water bottles"))

NameError: name 'llm' is not defined

## Memory

Memory refers to the mechanism that stores and manages the conversation history between a user and the AI. It helps maintain context and coherency throughout the interaction, enabling the AI to generate more relevant and accurate responses. Memory, such as ConversationBufferMemory, acts as a wrapper around ChatMessageHistory, extracting the messages and providing them to the chain for better context-aware generation.

In [3]:
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory

llm = OpenAI(model='gpt-3.5-turbo-instruct', temperature=0)
conversation=ConversationChain(llm=llm, verbose=True, memory=ConversationBufferMemory())

conversation.predict(input='Tell me about yourself')

conversation.predict(input='What can you do?')
conversation.predict(input='How can you help me with data analysis')

print(conversation)



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:

Human: Tell me about yourself
AI:

> Finished chain.


> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: Tell me about yourself
AI:  Well, I am an artificial intelligence created by a team of programmers and engineers. My purpose is to assist and interact with humans in various tasks and conversations. I am constantly learning and improving through algorithms and data analysis. I am currently ho

## Vector database

* create a vector db and add data to it
* create a QA chain 
* create tools that agent uses with retrieval QA cgain created as a tool
    * Tools are functions that perform specific duties, such as Google Search, database lookups
    * Tools are for agents to interact with the outside world
* create an agent that has all the tools
    * Agents decide which actions to take and in what order. 
    * Agent uses tools
        * **zero-shot-react-description** agent uses ReAct framework to decide which tool to employ based on purely tool descriptions. It necessiates description of the tool 
        * **react-docstore agent** engages search through document database. uses two tools - seach tool: searches for documents and look up tool: looks up exact terms in the document. 
        * **self-ask-with-search** employs single tool looking up responses for queries - like search api as tool
        * **conversation-react-description** agent is designed for conversational situations. Uses ReAct framework to select tool and uses memory to remember past conversations. 
        
* Invoke agent

In [5]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import DeepLake
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.chains import RetrievalQA

embeddings = OpenAIEmbeddings(model='text-embedding-ada-002')

texts = ['Napolean Bonaparte was born in 15 August 1769',
        'Louis XIV was born in 5 September 1638']

textsplitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
docs = textsplitter.create_documents(text)

ACTIVELOOP_ORG_ID = os.getenv('ACTIVELOOP_ORG_ID')
my_activeloop_dataset_name = "langchain_course"

dataset_path = f"hub://{ACTIVELOOP_ORG_ID}/{my_activeloop_dataset_name}"
db=DeepLake(dataset_path=dataset_path, embedding=embeddings)

db.add_documents(docs)

Deep Lake Dataset in hub://RC/langchain_course already exists, loading from the storage


Creating 111 embeddings in 1 batches of size 111:: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [01:13<00:00, 73.73s/it]

Dataset(path='hub://RC/langchain_course', tensors=['embedding', 'id', 'metadata', 'text'])

  tensor      htype       shape      dtype  compression
  -------    -------     -------    -------  ------- 
 embedding  embedding  (222, 1536)  float32   None   
    id        text      (222, 1)      str     None   
 metadata     json      (222, 1)      str     None   
   text       text      (222, 1)      str     None   


['a8c21a14-d8c0-11ee-82b0-acde48001122',
 'a8c21b04-d8c0-11ee-82b0-acde48001122',
 'a8c21b72-d8c0-11ee-82b0-acde48001122',
 'a8c21bb8-d8c0-11ee-82b0-acde48001122',
 'a8c21bfe-d8c0-11ee-82b0-acde48001122',
 'a8c21c44-d8c0-11ee-82b0-acde48001122',
 'a8c21c8a-d8c0-11ee-82b0-acde48001122',
 'a8c21cc6-d8c0-11ee-82b0-acde48001122',
 'a8c21d02-d8c0-11ee-82b0-acde48001122',
 'a8c21d3e-d8c0-11ee-82b0-acde48001122',
 'a8c21d84-d8c0-11ee-82b0-acde48001122',
 'a8c21dc0-d8c0-11ee-82b0-acde48001122',
 'a8c21dfc-d8c0-11ee-82b0-acde48001122',
 'a8c21e38-d8c0-11ee-82b0-acde48001122',
 'a8c21e74-d8c0-11ee-82b0-acde48001122',
 'a8c21eb0-d8c0-11ee-82b0-acde48001122',
 'a8c21eec-d8c0-11ee-82b0-acde48001122',
 'a8c21f28-d8c0-11ee-82b0-acde48001122',
 'a8c21f64-d8c0-11ee-82b0-acde48001122',
 'a8c21fa0-d8c0-11ee-82b0-acde48001122',
 'a8c21fdc-d8c0-11ee-82b0-acde48001122',
 'a8c22018-d8c0-11ee-82b0-acde48001122',
 'a8c22054-d8c0-11ee-82b0-acde48001122',
 'a8c22090-d8c0-11ee-82b0-acde48001122',
 'a8c220cc-d8c0-

In [6]:
# create a RetrievalQA chain:

retrieval_qa = RetrievalQA.from_chain_type(
llm=llm,
chain_type='stuff',
retriever=db.as_retriever())


# create an agent that uses the RetrievalQA chain as a tool:
from langchain.agents import initialize_agent, Tool, AgentType

tools = [
    Tool(name='Retrieval QA System',
        func=retrieval_qa.run,
        description='useful for answering questions')]
    
agent = initialize_agent(tools=tools, llm=llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
                        verbose=True)

response = agent.invoke('When was Napolean born')

/usr/local/lib/python3.9/site-packages/langchain_core/_api/deprecation.py:117: LangChainDeprecationWarning: The function `initialize_agent` was deprecated in LangChain 0.1.0 and will be removed in 0.2.0. Use Use new agent constructor methods like create_react_agent, create_json_agent, create_structured_chat_agent, etc. instead.
  warn_deprecated(




> Entering new AgentExecutor chain...
 I should use the Retrieval QA System to find the answer
Action: Retrieval QA System
Action Input: "When was Napolean born?"
Observation: 
I don't know.
Thought: I should try rephrasing the question
Action: Retrieval QA System
Action Input: "What is Napolean's birthdate?"
Observation:  I don't know.
Thought: I should try using a different source
Action: Retrieval QA System
Action Input: "When was Napolean born?"
Observation: 
I don't know.
Thought: I should try using a different source
Action: Retrieval QA System
Action Input: "When was Napolean born?"
Observation: 
I don't know.
Thought: I should try using a different source
Action: Retrieval QA System
Action Input: "When was Napolean born?"
Observation: 
I don't know.
Thought: I should try using a different source
Action: Retrieval QA System
Action Input: "When was Napolean born?"
Observation: 
I don't know.
Thought: I should try using a different source
Action: Retrieval QA System
Action Input

In [12]:
from langchain_community.utilities import GoogleSearchAPIWrapper
from langchain.agents import create_react_agent, create_self_ask_with_search_agent, AgentExecutor
from langchain import hub
from langchain.agents import Tool, AgentExecutor
from langchain_community.tools.tavily_search import TavilyAnswer

# search = GoogleSearchAPIWrapper(k=1)
search = TavilyAnswer(max_results=1)

# prompt = PromptTemplate(
#     input_variables=['Query'],
#     template='Write a summary of following text')

prompt = hub.pull("hwchase17/self-ask-with-search")

# def top5_results(query):
#     return search.results(query, 1)

tools = [TavilyAnswer(max_results=1, name="Intermediate Answer")]

# tools = [Tool(name='Intermediate Answer',
#              func=search.run,
#              description='useful for when to use google search to answer any questions')]


agent = create_self_ask_with_search_agent(tools=tools, llm=llm, prompt=prompt)
agentExecutor = AgentExecutor(agent=agent, tools=tools)

agentExecutor.invoke("What's the latest news about the Mars rover? Then please summarize the results.")

ValidationError: 1 validation error for TavilySearchAPIWrapper
__root__
  Did not find tavily_api_key, please add an environment variable `TAVILY_API_KEY` which contains it, or pass `tavily_api_key` as a named parameter. (type=value_error)